# Pseudo-Top Regime Mean Tuning

## 1. Purpose And Model

This notebook tunes the direct pseudo-top regime-mean model after the first version cleared the local packaging gate. The search stays close to the winning idea: pseudo-top probabilities define soft stratigraphic regimes, weighted Ridge means estimate each regime, and a global XGBoost mean provides a conservative fallback. The main changes are a denser Ridge-alpha sweep, several confidence schedules, and a few controlled global XGBoost variants.

## 2. Setup

In [1]:
"Configure imports, paths, and deterministic settings."

from pathlib import Path
import json
import shutil
import time

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import HuberRegressor, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor

pd.set_option("display.max_columns", 180)
pd.set_option("display.width", 220)

ROOT = Path.cwd()
if not (ROOT / "context").exists():
    ROOT = next(parent for parent in ROOT.parents if (parent / "context").exists())

RUN_DATE = "2026-07-01"
RUN_TAG = "pseudo_top_regime_mean_tuning"
RUN_ID = f"{RUN_DATE}__{RUN_TAG}"
SEED = 20260630
FORMATION_COLS = ["ANCC", "ASTNU", "ASTNL", "EGFDU", "EGFDL", "BUDA"]
PRIMARY_FOLD = "strict_hybrid_fold"
STRESS_FOLD = "analog_distance_fold"
MAX_TARGET_ROWS_PER_WELL = 120
ENSEMBLE_SEEDS = [20260630, 20260631]

DATA_DIR = ROOT / "data" / "raw"
TRAIN_DIR = DATA_DIR / "train"
TABLE_DIR = ROOT / "artifacts" / "tables"
REPORT_DIR = ROOT / "artifacts" / "reports"
RUN_DIR = ROOT / "artifacts" / "logs" / "runs" / RUN_ID
for directory in [TABLE_DIR, REPORT_DIR, RUN_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "run_id": RUN_ID,
    "seed": SEED,
    "formation_cols": FORMATION_COLS,
    "primary_fold": PRIMARY_FOLD,
    "stress_fold": STRESS_FOLD,
    "max_target_rows_per_well": MAX_TARGET_ROWS_PER_WELL,
    "ensemble_seeds": ENSEMBLE_SEEDS,
    "current_public_best_rmse": 14.472,
    "best_xgb_known_state_local_rmse": 13.4334,
    "best_xgb_known_state_stress_rmse": 13.7533,
    "promotion_margin": 0.05,
    "baseline_regime_mean_rmse": 13.092783707491265,
    "baseline_regime_mean_stress_rmse": 13.400574608144202,
}
with (RUN_DIR / "config.json").open("w", encoding="utf-8") as handle:
    json.dump(CONFIG, handle, indent=2)

print(ROOT)
print(RUN_DIR)

C:\Users\sardo\OneDrive\Desktop\Classes\Projects\Wellbore-tvt-Modeling
C:\Users\sardo\OneDrive\Desktop\Classes\Projects\Wellbore-tvt-Modeling\artifacts\logs\runs\2026-07-01__pseudo_top_regime_mean_tuning


## 3. Data And Fold-Safe Pseudo Tops

In [2]:
"Build sampled target-zone rows with legal predictors and direct top-elevation labels."

folds = pd.read_csv(TABLE_DIR / "validation__strict_cv_fold_assignments__2026-06-26.csv")
descriptors = pd.read_csv(TABLE_DIR / "model__typewell_alignment_feature_descriptors__2026-06-23.csv")
well_meta = folds.merge(descriptors, on="wellname", how="left")
well_meta = well_meta.drop_duplicates("wellname").reset_index(drop=True)

DESCRIPTOR_COLS = [
    "known_rows", "known_tvt_span", "known_md_span", "known_gr_mean_last300", "known_gr_std_last300", "known_gr_slope_last300",
    "xy_span_known", "z_drop_known", "typewell_corr_last300", "typewell_mae_last300", "typewell_best_lag_corr",
    "typewell_best_lag_tvt", "typewell_best_lag_mae", "selected_analog_distance", "extrapolation_stress_score",
    "analog_stress_score", "combined_stress_score", "descriptor_cluster",
]


def safe_divide(num, den):
    return num / (den + 1e-9)


def load_typewell(directory, wellname):
    path = Path(directory) / f"{wellname}__typewell.csv"
    if not path.exists():
        return pd.DataFrame(columns=["TVT", "GR", "Geology"])
    return pd.read_csv(path)


def interp_typewell_gr(typewell, tvt_values):
    if typewell.empty or not {"TVT", "GR"}.issubset(typewell.columns):
        return np.full(len(tvt_values), np.nan)
    clean = typewell[["TVT", "GR"]].dropna().sort_values("TVT")
    if len(clean) < 5:
        return np.full(len(tvt_values), np.nan)
    return np.interp(np.asarray(tvt_values, dtype=float), clean["TVT"].to_numpy(), clean["GR"].to_numpy(), left=np.nan, right=np.nan)


def build_target_rows(csv_path, meta_row):
    df = pd.read_csv(csv_path).sort_values("MD").reset_index(drop=True)
    known_mask = df["TVT_input"].notna()
    if known_mask.sum() == 0 or (~known_mask).sum() == 0:
        return pd.DataFrame()
    last_known_idx = int(np.flatnonzero(known_mask.to_numpy())[-1])
    known = df.loc[:last_known_idx].copy()
    target = df.loc[last_known_idx + 1:].copy()
    target = target.loc[target["TVT"].notna()].copy()
    if target.empty:
        return pd.DataFrame()
    if len(target) > MAX_TARGET_ROWS_PER_WELL:
        sample_idx = np.linspace(0, len(target) - 1, MAX_TARGET_ROWS_PER_WELL).round().astype(int)
        target = target.iloc[np.unique(sample_idx)].copy()

    typewell = load_typewell(csv_path.parent, meta_row["wellname"])
    last = known.iloc[-1]
    first = df.iloc[0]
    known_gr = known["GR"].astype(float).clip(0, 300)
    known_gr_finite = known_gr.dropna()
    gr_fill = float(known_gr.tail(300).median()) if known_gr.notna().any() else 0.0
    target_gr = target["GR"].astype(float).clip(0, 300).fillna(gr_fill)

    known_md_span_state = float(known["MD"].iloc[-1] - known["MD"].iloc[0]) if len(known) > 1 else 0.0
    known_tvt_delta_state = float(known["TVT_input"].iloc[-1] - known["TVT_input"].iloc[0]) if len(known) > 1 else 0.0
    known_tvt_slope_state = known_tvt_delta_state / max(abs(known_md_span_state), 1.0)
    known_xy_span_state = float(np.hypot(known["X"].iloc[-1] - known["X"].iloc[0], known["Y"].iloc[-1] - known["Y"].iloc[0])) if len(known) > 1 else 0.0
    known_z_drop_state = float(known["Z"].iloc[-1] - known["Z"].iloc[0]) if len(known) > 1 else 0.0
    known_gr_coverage_state = float(known_gr.notna().mean()) if len(known_gr) else 0.0
    known_gr_last_state = float(known_gr_finite.iloc[-1]) if len(known_gr_finite) else gr_fill
    known_gr_iqr_state = float(known_gr_finite.quantile(0.75) - known_gr_finite.quantile(0.25)) if len(known_gr_finite) >= 4 else 0.0

    target = target.reset_index(drop=False).rename(columns={"index": "row_idx_original"})
    out = pd.DataFrame({
        "wellname": meta_row["wellname"],
        "row_idx_original": target["row_idx_original"].astype(float),
        "MD": target["MD"].astype(float),
        "X": target["X"].astype(float),
        "Y": target["Y"].astype(float),
        "Z": target["Z"].astype(float),
        "GR": target_gr.to_numpy(),
        "TVT": target["TVT"].astype(float),
        "last_known_tvt_feature": float(last["TVT"]),
        "last_known_md": float(last["MD"]),
        "last_known_x": float(last["X"]),
        "last_known_y": float(last["Y"]),
        "last_known_z": float(last["Z"]),
        "first_md": float(first["MD"]),
        "state_known_tvt_last": float(last["TVT_input"] if pd.notna(last.get("TVT_input", np.nan)) else last["TVT"]),
        "state_known_tvt_slope": known_tvt_slope_state,
        "state_known_gr_coverage": known_gr_coverage_state,
        "state_known_gr_last": known_gr_last_state,
        "state_known_gr_iqr": known_gr_iqr_state,
        "state_known_xy_span": known_xy_span_state,
        "state_known_z_drop": known_z_drop_state,
        "state_known_md_span": known_md_span_state,
    })
    out["target_delta"] = out["TVT"] - out["last_known_tvt_feature"]
    out["md_from_last_known"] = out["MD"] - out["last_known_md"]
    out["md_from_start"] = out["MD"] - out["first_md"]
    out["eval_progress"] = safe_divide(np.arange(1, len(out) + 1), len(out))
    out["eval_progress_sq"] = out["eval_progress"] ** 2
    out["dx_from_last"] = out["X"] - out["last_known_x"]
    out["dy_from_last"] = out["Y"] - out["last_known_y"]
    out["dz_from_last"] = out["Z"] - out["last_known_z"]
    out["horizontal_from_last"] = np.sqrt(out["dx_from_last"] ** 2 + out["dy_from_last"] ** 2)
    out["distance_from_last"] = np.sqrt(out["horizontal_from_last"] ** 2 + out["dz_from_last"] ** 2)
    out["distance_log1p"] = np.log1p(out["distance_from_last"].clip(lower=0))
    out["tvt_linear_proxy"] = out["last_known_tvt_feature"] + out["md_from_last_known"]
    best_lag = float(meta_row.get("typewell_best_lag_tvt", 0.0) if pd.notna(meta_row.get("typewell_best_lag_tvt", np.nan)) else 0.0)
    out["typewell_gr_linear_tvt"] = interp_typewell_gr(typewell, out["tvt_linear_proxy"].to_numpy())
    out["typewell_gr_lag_best"] = interp_typewell_gr(typewell, out["tvt_linear_proxy"].to_numpy() + best_lag)
    out["gr_minus_typewell_linear"] = out["GR"] - out["typewell_gr_linear_tvt"]
    out["gr_minus_typewell_bestlag"] = out["GR"] - out["typewell_gr_lag_best"]

    for formation in FORMATION_COLS:
        out[f"formation_top_{formation}"] = target[formation].astype(float).to_numpy()
        out[f"label_signed_offset_{formation}"] = out["Z"] - out[f"formation_top_{formation}"]
        out[f"label_abs_offset_{formation}"] = np.abs(out[f"label_signed_offset_{formation}"])

    abs_label_cols = [f"label_abs_offset_{formation}" for formation in FORMATION_COLS]
    out["label_nearest_abs_offset"] = out[abs_label_cols].min(axis=1)
    out["label_nearest_formation"] = out[abs_label_cols].idxmin(axis=1).str.replace("label_abs_offset_", "", regex=False)
    out["label_mean_abs_offset"] = out[abs_label_cols].mean(axis=1)

    for col in [PRIMARY_FOLD, STRESS_FOLD, "descriptor_cluster_fold", "extrapolation_fold"]:
        out[col] = int(meta_row[col])
    for col in DESCRIPTOR_COLS:
        out[col] = float(meta_row[col]) if pd.notna(meta_row[col]) else np.nan
    return out


frames = []
for meta_row in well_meta.to_dict(orient="records"):
    csv_path = TRAIN_DIR / f"{meta_row['wellname']}__horizontal_well.csv"
    if csv_path.exists():
        frame = build_target_rows(csv_path, meta_row)
        if not frame.empty:
            frames.append(frame)

data = pd.concat(frames, ignore_index=True)

SAFE_ROW_COLS = [
    "row_idx_original", "MD", "X", "Y", "Z", "GR", "last_known_tvt_feature", "last_known_md",
    "md_from_last_known", "md_from_start", "eval_progress", "eval_progress_sq",
    "dx_from_last", "dy_from_last", "dz_from_last", "horizontal_from_last", "distance_from_last", "distance_log1p",
]
SAFE_FEATURE_COLS = [col for col in [*SAFE_ROW_COLS, *DESCRIPTOR_COLS] if col in data.columns]
TYPEWELL_FEATURE_COLS = [
    "tvt_linear_proxy", "typewell_gr_linear_tvt", "typewell_gr_lag_best", "gr_minus_typewell_linear", "gr_minus_typewell_bestlag",
]
TYPEWELL_FEATURE_COLS = [col for col in TYPEWELL_FEATURE_COLS if col in data.columns]
STATE_FEATURE_COLS = [
    "state_known_tvt_last", "state_known_tvt_slope", "state_known_gr_coverage", "state_known_gr_last",
    "state_known_gr_iqr", "state_known_xy_span", "state_known_z_drop", "state_known_md_span",
]
DIRECT_FORMATION_FEATURE_COLS = [
    *[f"label_signed_offset_{formation}" for formation in FORMATION_COLS],
    *[f"label_abs_offset_{formation}" for formation in FORMATION_COLS],
    "label_nearest_abs_offset", "label_mean_abs_offset",
]

top_label_summary = []
for formation in FORMATION_COLS:
    values = data[f"formation_top_{formation}"].astype(float)
    offsets = data[f"label_signed_offset_{formation}"].astype(float)
    top_label_summary.append({
        "formation": formation,
        "top_mean": float(values.mean()),
        "top_std": float(values.std()),
        "top_min": float(values.min()),
        "top_max": float(values.max()),
        "signed_offset_mean": float(offsets.mean()),
        "signed_offset_std": float(offsets.std()),
        "abs_offset_mean": float(np.abs(offsets).mean()),
    })
top_label_summary = pd.DataFrame(top_label_summary)
label_counts = data["label_nearest_formation"].value_counts().rename_axis("formation").reset_index(name="nearest_label_rows")
top_label_summary = top_label_summary.merge(label_counts, on="formation", how="left").fillna({"nearest_label_rows": 0})

label_path = TABLE_DIR / "model__pseudo_top_regime_mean_tuning_top_label_summary__2026-07-01.csv"
top_label_summary.to_csv(label_path, index=False)
shutil.copy2(label_path, RUN_DIR / label_path.name)

feature_preview_path = TABLE_DIR / "model__pseudo_top_regime_mean_tuning_feature_preview__2026-07-01.csv"
data.head(500).to_csv(feature_preview_path, index=False)
shutil.copy2(feature_preview_path, RUN_DIR / feature_preview_path.name)

print(data.shape)
print("safe_features", len(SAFE_FEATURE_COLS))
print(top_label_summary.to_string(index=False))

(92760, 81)
safe_features 36
formation     top_mean    top_std   top_min  top_max  signed_offset_mean  signed_offset_std  abs_offset_mean  nearest_label_rows
     ANCC -9008.099049 633.914859 -10647.31 -7301.76         -416.172460          55.411650       416.172460                 0.0
    ASTNU -9190.780447 639.418693 -10852.79 -7509.20         -244.653750          63.560087       244.653750                 0.0
    ASTNL -9255.299324 624.857096 -10865.58 -7614.43         -180.134874          51.475069       180.135674              2455.0
    EGFDU -9349.765034 619.827567 -10887.58 -7704.03          -85.669164          43.889718        91.053770              3919.0
    EGFDL -9391.937722 619.776843 -10932.35 -7731.61          -43.798726          44.572447        55.375403             50858.0
     BUDA -9516.381964 633.862561 -11104.73 -7835.57           80.947766          45.807212        80.953400             35528.0


## 4. Pseudo-Top Probabilities

In [3]:
"Train fold-safe top-elevation ensembles and derive uncertainty, margin, and nearest-top features."

PSEUDO_TOP_PREDICTOR_COLS = [
    *SAFE_FEATURE_COLS,
    *TYPEWELL_FEATURE_COLS,
    *STATE_FEATURE_COLS,
]
PSEUDO_TOP_PREDICTOR_COLS = list(dict.fromkeys([col for col in PSEUDO_TOP_PREDICTOR_COLS if col in data.columns]))

pseudo_params = {
    "objective": "reg:squarederror",
    "n_estimators": 220,
    "learning_rate": 0.040,
    "max_depth": 3,
    "min_child_weight": 90,
    "subsample": 0.85,
    "colsample_bytree": 0.85,
    "reg_alpha": 0.20,
    "reg_lambda": 10.0,
    "tree_method": "hist",
    "n_jobs": 4,
    "verbosity": 0,
}


def rmse(y_true, y_pred):
    return float(mean_squared_error(y_true, y_pred) ** 0.5)


def softmax_neg_abs(abs_matrix, scale=50.0):
    logits = -abs_matrix / scale
    logits = logits - np.nanmax(logits, axis=1, keepdims=True)
    exp_logits = np.exp(logits)
    return exp_logits / np.clip(exp_logits.sum(axis=1, keepdims=True), 1e-12, None)


def add_pseudo_features_for_fold_column(frame, fold_col):
    out = frame.copy()
    proxy_quality_rows = []
    for formation in FORMATION_COLS:
        out[f"pseudo_top_elev_{formation}__{fold_col}"] = np.nan
        out[f"pseudo_top_std_{formation}__{fold_col}"] = np.nan

    X_all = out[PSEUDO_TOP_PREDICTOR_COLS].replace([np.inf, -np.inf], np.nan)
    for fold in sorted(out[fold_col].dropna().unique()):
        valid_mask = out[fold_col].eq(fold)
        train_mask = ~valid_mask
        X_train = X_all.loc[train_mask]
        X_valid = X_all.loc[valid_mask]
        for formation in FORMATION_COLS:
            target_col = f"formation_top_{formation}"
            label_train_mask = train_mask & out[target_col].notna()
            label_valid_mask = valid_mask & out[target_col].notna()
            seed_predictions = []
            if label_train_mask.sum() >= 20:
                X_label_train = X_all.loc[label_train_mask]
                y_train = out.loc[label_train_mask, target_col].astype(float)
                for seed in ENSEMBLE_SEEDS:
                    model = XGBRegressor(**pseudo_params, random_state=seed)
                    model.fit(X_label_train, y_train, verbose=False)
                    seed_predictions.append(model.predict(X_valid))
                pred_matrix = np.column_stack(seed_predictions)
                pred_mean = pred_matrix.mean(axis=1)
                pred_std = pred_matrix.std(axis=1)
            else:
                pred_mean = np.full(valid_mask.sum(), float(out[target_col].median()))
                pred_std = np.zeros(valid_mask.sum())
            out.loc[valid_mask, f"pseudo_top_elev_{formation}__{fold_col}"] = pred_mean
            out.loc[valid_mask, f"pseudo_top_std_{formation}__{fold_col}"] = pred_std
            if label_valid_mask.sum() > 0:
                valid_index = out.loc[valid_mask].index
                pred_series = pd.Series(pred_mean, index=valid_index).loc[label_valid_mask]
                y_valid = out.loc[label_valid_mask, target_col].astype(float)
                true_offset = out.loc[label_valid_mask, "Z"].to_numpy(float) - y_valid.to_numpy(float)
                pred_offset = out.loc[label_valid_mask, "Z"].to_numpy(float) - pred_series.to_numpy(float)
                corr = float(np.corrcoef(y_valid.to_numpy(float), pred_series.to_numpy(float))[0, 1]) if len(y_valid) > 2 else np.nan
                top_rmse = rmse(y_valid, pred_series)
                top_mae = float(mean_absolute_error(y_valid, pred_series))
                signed_offset_rmse = rmse(true_offset, pred_offset)
            else:
                corr = np.nan
                top_rmse = np.nan
                top_mae = np.nan
                signed_offset_rmse = np.nan
            proxy_quality_rows.append({
                "fold_column": fold_col,
                "fold": int(fold),
                "formation": formation,
                "n_train": int(label_train_mask.sum()),
                "n_valid": int(label_valid_mask.sum()),
                "top_rmse": top_rmse,
                "top_mae": top_mae,
                "top_corr": corr,
                "signed_offset_rmse": signed_offset_rmse,
                "mean_ensemble_std": float(np.mean(pred_std)),
            })

    for formation in FORMATION_COLS:
        top_col = f"pseudo_top_elev_{formation}__{fold_col}"
        std_col = f"pseudo_top_std_{formation}__{fold_col}"
        out[top_col] = out[top_col].fillna(float(out[f"formation_top_{formation}"].median()))
        out[std_col] = out[std_col].fillna(float(out[std_col].median()) if out[std_col].notna().any() else 0.0)
        out[f"pseudo_signed_offset_{formation}__{fold_col}"] = out["Z"] - out[top_col]
        out[f"pseudo_abs_offset_{formation}__{fold_col}"] = np.abs(out[f"pseudo_signed_offset_{formation}__{fold_col}"])

    abs_cols = [f"pseudo_abs_offset_{formation}__{fold_col}" for formation in FORMATION_COLS]
    std_cols = [f"pseudo_top_std_{formation}__{fold_col}" for formation in FORMATION_COLS]
    abs_matrix = out[abs_cols].to_numpy(float)
    probs = softmax_neg_abs(abs_matrix)
    sorted_abs = np.sort(abs_matrix, axis=1)
    sorted_probs = np.sort(probs, axis=1)[:, ::-1]
    nearest_idx = np.argmin(abs_matrix, axis=1)

    out[f"pseudo_nearest_abs_offset__{fold_col}"] = sorted_abs[:, 0]
    out[f"pseudo_second_abs_offset__{fold_col}"] = sorted_abs[:, 1]
    out[f"pseudo_top_margin__{fold_col}"] = out[f"pseudo_second_abs_offset__{fold_col}"] - out[f"pseudo_nearest_abs_offset__{fold_col}"]
    out[f"pseudo_top_confidence__{fold_col}"] = 1.0 / (1.0 + np.exp(-out[f"pseudo_top_margin__{fold_col}"].clip(-100, 100) / 25.0))
    out[f"pseudo_near_boundary_25__{fold_col}"] = (out[f"pseudo_nearest_abs_offset__{fold_col}"] <= 25.0).astype(float)
    out[f"pseudo_uncertain_boundary__{fold_col}"] = ((out[f"pseudo_nearest_abs_offset__{fold_col}"] <= 50.0) & (out[f"pseudo_top_margin__{fold_col}"] <= 25.0)).astype(float)
    out[f"pseudo_mean_abs_offset__{fold_col}"] = abs_matrix.mean(axis=1)
    out[f"pseudo_min_std__{fold_col}"] = out[std_cols].min(axis=1)
    out[f"pseudo_mean_std__{fold_col}"] = out[std_cols].mean(axis=1)
    out[f"pseudo_top1_prob__{fold_col}"] = sorted_probs[:, 0]
    out[f"pseudo_top2_prob__{fold_col}"] = sorted_probs[:, 1]
    out[f"pseudo_top_entropy__{fold_col}"] = -(probs * np.log(probs + 1e-12)).sum(axis=1)
    out[f"pseudo_nearest_formation__{fold_col}"] = [FORMATION_COLS[int(idx)] for idx in nearest_idx]
    out[f"pseudo_nearest_matches_label__{fold_col}"] = (out[f"pseudo_nearest_formation__{fold_col}"] == out["label_nearest_formation"]).astype(float)

    return out, pd.DataFrame(proxy_quality_rows)


pseudo_feature_cols_by_fold = {}
pseudo_uncertainty_cols_by_fold = {}
quality_parts = []
for fold_col in [PRIMARY_FOLD, STRESS_FOLD]:
    data, quality = add_pseudo_features_for_fold_column(data, fold_col)
    quality_parts.append(quality)
    pseudo_feature_cols_by_fold[fold_col] = [
        *[f"pseudo_signed_offset_{formation}__{fold_col}" for formation in FORMATION_COLS],
        *[f"pseudo_abs_offset_{formation}__{fold_col}" for formation in FORMATION_COLS],
        f"pseudo_nearest_abs_offset__{fold_col}",
        f"pseudo_second_abs_offset__{fold_col}",
        f"pseudo_mean_abs_offset__{fold_col}",
    ]
    pseudo_uncertainty_cols_by_fold[fold_col] = [
        *[f"pseudo_top_std_{formation}__{fold_col}" for formation in FORMATION_COLS],
        f"pseudo_top_margin__{fold_col}",
        f"pseudo_top_confidence__{fold_col}",
        f"pseudo_near_boundary_25__{fold_col}",
        f"pseudo_uncertain_boundary__{fold_col}",
        f"pseudo_min_std__{fold_col}",
        f"pseudo_mean_std__{fold_col}",
        f"pseudo_top1_prob__{fold_col}",
        f"pseudo_top2_prob__{fold_col}",
        f"pseudo_top_entropy__{fold_col}",
    ]

proxy_quality = pd.concat(quality_parts, ignore_index=True)
nearest_quality = []
for fold_col in [PRIMARY_FOLD, STRESS_FOLD]:
    nearest_quality.append({
        "fold_column": fold_col,
        "fold": "all",
        "formation": "nearest_classifier",
        "n_train": np.nan,
        "n_valid": int(len(data)),
        "top_rmse": np.nan,
        "top_mae": np.nan,
        "top_corr": np.nan,
        "signed_offset_rmse": np.nan,
        "mean_ensemble_std": float(data[f"pseudo_mean_std__{fold_col}"].mean()),
        "nearest_accuracy": float(data[f"pseudo_nearest_matches_label__{fold_col}"].mean()),
    })
proxy_quality["nearest_accuracy"] = np.nan
proxy_quality = pd.concat([proxy_quality, pd.DataFrame(nearest_quality)], ignore_index=True)

proxy_quality_path = TABLE_DIR / "model__pseudo_top_regime_mean_tuning_proxy_quality__2026-07-01.csv"
proxy_quality.to_csv(proxy_quality_path, index=False)
shutil.copy2(proxy_quality_path, RUN_DIR / proxy_quality_path.name)
proxy_quality.groupby(["fold_column", "formation"], dropna=False).agg(top_rmse=("top_rmse", "mean"), top_mae=("top_mae", "mean"), mean_ensemble_std=("mean_ensemble_std", "mean"), nearest_accuracy=("nearest_accuracy", "mean")).reset_index()

,fold_column,formation,top_rmse,top_mae,mean_ensemble_std,nearest_accuracy
0,analog_distance_fold,ANCC,36.448121,27.561076,2.225590,NaN
1,analog_distance_fold,ASTNL,36.512561,27.158327,2.604606,NaN
2,analog_distance_fold,ASTNU,38.155576,28.416605,2.408923,NaN
3,analog_distance_fold,BUDA,34.737212,25.140603,2.262772,NaN
4,analog_distance_fold,EGFDL,34.746888,25.533790,2.353502,NaN
5,analog_distance_fold,EGFDU,34.494617,25.287956,2.264115,NaN
6,analog_distance_fold,nearest_classifier,NaN,NaN,2.353398,0.579743
7,strict_hybrid_fold,ANCC,35.638111,26.984800,1.995119,NaN
8,strict_hybrid_fold,ASTNL,35.637506,26.891537,2.281886,NaN
9,strict_hybrid_fold,ASTNU,37.183606,27.877467,2.611257,NaN


## 5. Regime Probability Basis

In [4]:
"Create soft formation-regime probabilities and MAP residual basis features."


def add_probability_and_basis_features(frame, fold_col, scale=50.0):
    out = frame
    abs_cols = [f"pseudo_abs_offset_{formation}__{fold_col}" for formation in FORMATION_COLS]
    abs_matrix = out[abs_cols].to_numpy(float)
    logits = -abs_matrix / scale
    logits = logits - np.nanmax(logits, axis=1, keepdims=True)
    probs = np.exp(logits)
    probs = probs / np.clip(probs.sum(axis=1, keepdims=True), 1e-12, None)
    for idx, formation in enumerate(FORMATION_COLS):
        prob_col = f"mix_prob_{formation}__{fold_col}"
        out[prob_col] = probs[:, idx]
        out[f"mix_prob_{formation}_x_progress__{fold_col}"] = out[prob_col] * out["eval_progress"]
        out[f"mix_prob_{formation}_x_distance__{fold_col}"] = out[prob_col] * out["distance_log1p"]
        out[f"mix_prob_{formation}_x_near_top__{fold_col}"] = out[prob_col] * (out[f"pseudo_abs_offset_{formation}__{fold_col}"] <= 40.0).astype(float)
    out[f"mix_entropy__{fold_col}"] = -(probs * np.log(probs + 1e-12)).sum(axis=1)
    out[f"mix_top1_prob__{fold_col}"] = np.sort(probs, axis=1)[:, -1]
    out[f"mix_top2_prob__{fold_col}"] = np.sort(probs, axis=1)[:, -2]
    out[f"mix_prob_margin__{fold_col}"] = out[f"mix_top1_prob__{fold_col}"] - out[f"mix_top2_prob__{fold_col}"]
    out[f"mix_confidence__{fold_col}"] = 1.0 / (1.0 + np.exp((out[f"mix_entropy__{fold_col}"] - 1.15) / 0.12))
    return out


mix_basis_cols_by_fold = {}
mix_uncertainty_cols_by_fold = {}
for active_fold in [PRIMARY_FOLD, STRESS_FOLD]:
    data = add_probability_and_basis_features(data, active_fold)
    mix_basis_cols_by_fold[active_fold] = []
    for formation in FORMATION_COLS:
        mix_basis_cols_by_fold[active_fold].extend([
            f"mix_prob_{formation}__{active_fold}",
            f"mix_prob_{formation}_x_progress__{active_fold}",
            f"mix_prob_{formation}_x_distance__{active_fold}",
            f"mix_prob_{formation}_x_near_top__{active_fold}",
        ])
    mix_uncertainty_cols_by_fold[active_fold] = [
        f"mix_entropy__{active_fold}",
        f"mix_top1_prob__{active_fold}",
        f"mix_top2_prob__{active_fold}",
        f"mix_prob_margin__{active_fold}",
        f"mix_confidence__{active_fold}",
        f"pseudo_top_margin__{active_fold}",
        f"pseudo_mean_std__{active_fold}",
        f"pseudo_top_entropy__{active_fold}",
    ]

print(len(mix_basis_cols_by_fold[PRIMARY_FOLD]), len(mix_uncertainty_cols_by_fold[PRIMARY_FOLD]))
data[mix_basis_cols_by_fold[PRIMARY_FOLD] + mix_uncertainty_cols_by_fold[PRIMARY_FOLD]].head()

24 8


,mix_prob_ANCC__strict_hybrid_fold,mix_prob_ANCC_x_progress__strict_hybrid_fold,mix_prob_ANCC_x_distance__strict_hybrid_fold,mix_prob_ANCC_x_near_top__strict_hybrid_fold,mix_prob_ASTNU__strict_hybrid_fold,mix_prob_ASTNU_x_progress__strict_hybrid_fold,mix_prob_ASTNU_x_distance__strict_hybrid_fold,mix_prob_ASTNU_x_near_top__strict_hybrid_fold,mix_prob_ASTNL__strict_hybrid_fold,mix_prob_ASTNL_x_progress__strict_hybrid_fold,mix_prob_ASTNL_x_distance__strict_hybrid_fold,mix_prob_ASTNL_x_near_top__strict_hybrid_fold,mix_prob_EGFDU__strict_hybrid_fold,mix_prob_EGFDU_x_progress__strict_hybrid_fold,mix_prob_EGFDU_x_distance__strict_hybrid_fold,mix_prob_EGFDU_x_near_top__strict_hybrid_fold,mix_prob_EGFDL__strict_hybrid_fold,mix_prob_EGFDL_x_progress__strict_hybrid_fold,mix_prob_EGFDL_x_distance__strict_hybrid_fold,mix_prob_EGFDL_x_near_top__strict_hybrid_fold,mix_prob_BUDA__strict_hybrid_fold,mix_prob_BUDA_x_progress__strict_hybrid_fold,mix_prob_BUDA_x_distance__strict_hybrid_fold,mix_prob_BUDA_x_near_top__strict_hybrid_fold,mix_entropy__strict_hybrid_fold,mix_top1_prob__strict_hybrid_fold,mix_top2_prob__strict_hybrid_fold,mix_prob_margin__strict_hybrid_fold,mix_confidence__strict_hybrid_fold,pseudo_top_margin__strict_hybrid_fold,pseudo_mean_std__strict_hybrid_fold,pseudo_top_entropy__strict_hybrid_fold
0,0.000478,0.000004,0.000331,0.0,0.019256,0.000160,0.013352,0.0,0.022512,0.000188,0.015610,0.0,0.128967,0.001075,0.089425,0.0,0.263025,0.002192,0.182381,0.0,0.565762,0.004715,0.392298,0.0,1.102786,0.565762,0.263025,0.302737,0.597114,38.296250,2.545573,1.102786
1,0.000483,0.000008,0.001702,0.0,0.019453,0.000324,0.068595,0.0,0.022741,0.000379,0.080193,0.0,0.130281,0.002171,0.459411,0.0,0.266382,0.004440,0.939341,0.0,0.560661,0.009344,1.977057,0.0,1.108685,0.560661,0.266382,0.294279,0.585233,37.209297,2.524414,1.108685
2,0.000491,0.000012,0.002058,0.0,0.019798,0.000495,0.082946,0.0,0.023145,0.000579,0.096970,0.0,0.132594,0.003315,0.555521,0.0,0.271110,0.006778,1.135855,0.0,0.552863,0.013822,2.316302,0.0,1.117970,0.552863,0.271110,0.281753,0.566337,35.629297,2.524414,1.117970
3,0.000502,0.000017,0.002308,0.0,0.020246,0.000675,0.093032,0.0,0.023669,0.000789,0.108761,0.0,0.135598,0.004520,0.623074,0.0,0.276549,0.009218,1.270747,0.0,0.543436,0.018115,2.497099,0.0,1.129190,0.543436,0.276549,0.266887,0.543246,33.776250,2.545573,1.129190
4,0.000514,0.000021,0.002504,0.0,0.020709,0.000863,0.100961,0.0,0.024211,0.001009,0.118031,0.0,0.138698,0.005779,0.676176,0.0,0.282871,0.011786,1.379048,0.0,0.532998,0.022208,2.598458,0.0,1.140844,0.532998,0.282871,0.250126,0.519065,31.676250,2.545573,1.140844


## 6. Regime-Specific Mean Models

In [5]:

"Fit tuned global means, regime-specific MAP means, and probability-weighted blends."

BASE_FEATURE_COLS = list(dict.fromkeys([*SAFE_FEATURE_COLS, *TYPEWELL_FEATURE_COLS]))
REGIME_FEATURE_COLS = list(dict.fromkeys([
    *SAFE_FEATURE_COLS,
    *TYPEWELL_FEATURE_COLS,
    "state_known_tvt_slope", "state_known_gr_coverage", "state_known_gr_last", "state_known_gr_iqr",
    "state_known_xy_span", "state_known_z_drop", "state_known_md_span",
]))
REGIME_FEATURE_COLS = [col for col in REGIME_FEATURE_COLS if col in data.columns]

mean_models = [
    "xgb_square_global",
    "xgb_square_slow_global",
]
ridge_alphas = [40.0, 80.0, 140.0, 220.0]
blend_schedules = {
    "s050": {"kind": "fixed", "weight": 0.50},
    "s065": {"kind": "fixed", "weight": 0.65},
    "confidence_s100": {"kind": "confidence", "max_weight": 1.0},
    "confidence_pow050_s100": {"kind": "confidence_power", "max_weight": 1.0, "power": 0.50},
    "margin_heavy_s100": {"kind": "margin_heavy", "max_weight": 1.0},
}


def rmse(y_true, y_pred):
    return float(mean_squared_error(y_true, y_pred) ** 0.5)


def metric_row(actual, pred, baseline):
    return {
        "rmse": rmse(actual, pred),
        "mae": float(mean_absolute_error(actual, pred)),
        "r2": float(r2_score(actual, pred)),
        "last_known_hold_rmse": rmse(actual, baseline),
    }


def fit_global_mean(name, X_train, y_train, X_valid):
    if name == "xgb_square_global":
        model = XGBRegressor(
            objective="reg:squarederror", n_estimators=520, learning_rate=0.036, max_depth=5,
            min_child_weight=90, subsample=0.85, colsample_bytree=0.85, reg_alpha=0.20,
            reg_lambda=8.0, gamma=0.10, tree_method="hist", random_state=SEED, n_jobs=4, verbosity=0,
        )
    elif name == "xgb_square_slow_global":
        model = XGBRegressor(
            objective="reg:squarederror", n_estimators=760, learning_rate=0.026, max_depth=4,
            min_child_weight=80, subsample=0.88, colsample_bytree=0.88, reg_alpha=0.15,
            reg_lambda=7.0, gamma=0.05, tree_method="hist", random_state=SEED, n_jobs=4, verbosity=0,
        )
    elif name == "xgb_square_shallow_global":
        model = XGBRegressor(
            objective="reg:squarederror", n_estimators=680, learning_rate=0.030, max_depth=3,
            min_child_weight=70, subsample=0.90, colsample_bytree=0.90, reg_alpha=0.10,
            reg_lambda=5.0, gamma=0.00, tree_method="hist", random_state=SEED, n_jobs=4, verbosity=0,
        )
    elif name == "xgb_pseudohuber_global":
        model = XGBRegressor(
            objective="reg:pseudohubererror", n_estimators=650, learning_rate=0.032, max_depth=4,
            min_child_weight=100, subsample=0.85, colsample_bytree=0.85, reg_alpha=0.15,
            reg_lambda=7.0, gamma=0.05, tree_method="hist", random_state=SEED, n_jobs=4, verbosity=0,
        )
    else:
        raise ValueError(name)
    model.fit(X_train, y_train, verbose=False)
    return model.predict(X_train), model.predict(X_valid), model


def fit_weighted_ridge(X_train, y_train, sample_weight, alpha):
    imputer = SimpleImputer(strategy="median")
    scaler = StandardScaler()
    ridge = Ridge(alpha=alpha)
    X_imp = imputer.fit_transform(X_train)
    X_scaled = scaler.fit_transform(X_imp)
    ridge.fit(X_scaled, y_train, sample_weight=sample_weight)
    return {"imputer": imputer, "scaler": scaler, "ridge": ridge}


def predict_weighted_ridge(model, X):
    return model["ridge"].predict(model["scaler"].transform(model["imputer"].transform(X)))


def confidence_components(frame, fold_col):
    confidence = frame[f"mix_confidence__{fold_col}"].fillna(frame[f"mix_confidence__{fold_col}"].median()).clip(0.0, 1.0)
    margin = frame[f"mix_prob_margin__{fold_col}"].fillna(frame[f"mix_prob_margin__{fold_col}"].median()).clip(0.0, 1.0)
    entropy = frame[f"mix_entropy__{fold_col}"].fillna(frame[f"mix_entropy__{fold_col}"].median())
    entropy_score = 1.0 / (1.0 + np.exp((entropy - 1.15) / 0.12))
    return np.asarray(confidence), np.asarray(margin), np.asarray(entropy_score)


def blend_weights(frame, fold_col, schedule):
    if schedule["kind"] == "fixed":
        return np.full(len(frame), float(schedule["weight"]))
    confidence, margin, entropy_score = confidence_components(frame, fold_col)
    if schedule["kind"] == "confidence":
        raw = 0.50 * confidence + 0.35 * margin + 0.15 * entropy_score
    elif schedule["kind"] == "confidence_power":
        raw = 0.50 * confidence + 0.35 * margin + 0.15 * entropy_score
        raw = np.power(np.clip(raw, 0.0, 1.0), float(schedule["power"]))
    elif schedule["kind"] == "margin_heavy":
        raw = 0.30 * confidence + 0.55 * margin + 0.15 * entropy_score
    elif schedule["kind"] == "entropy_guard":
        raw = (0.55 * confidence + 0.45 * margin) * np.clip(entropy_score, 0.05, 1.0)
    else:
        raise ValueError(schedule["kind"])
    return np.clip(schedule["max_weight"] * raw, 0.0, schedule["max_weight"])


def evaluate_regime_mean(fold_col):
    prob_cols = [f"mix_prob_{formation}__{fold_col}" for formation in FORMATION_COLS]
    X_base = data[BASE_FEATURE_COLS].replace([np.inf, -np.inf], np.nan)
    X_regime = data[REGIME_FEATURE_COLS].replace([np.inf, -np.inf], np.nan)
    y_delta = data["target_delta"].astype(float)
    y_tvt = data["TVT"].astype(float)
    baseline = data["last_known_tvt_feature"].astype(float)
    fold_rows = []
    pred_parts = []
    diag_rows = []
    for fold in sorted(data[fold_col].dropna().unique()):
        valid_mask = data[fold_col].eq(fold)
        train_mask = ~valid_mask
        actual = y_tvt.loc[valid_mask].to_numpy()
        base_tvt = baseline.loc[valid_mask].to_numpy()
        train_probs = data.loc[train_mask, prob_cols].to_numpy(float)
        valid_probs = data.loc[valid_mask, prob_cols].to_numpy(float)
        for global_name in mean_models:
            global_train_delta, global_valid_delta, global_model = fit_global_mean(global_name, X_base.loc[train_mask], y_delta.loc[train_mask], X_base.loc[valid_mask])
            for alpha in ridge_alphas:
                regime_train_preds = []
                regime_valid_preds = []
                for idx, formation in enumerate(FORMATION_COLS):
                    weights = np.clip(train_probs[:, idx], 1e-4, None)
                    weights = weights / np.mean(weights)
                    model = fit_weighted_ridge(X_regime.loc[train_mask], y_delta.loc[train_mask], weights, alpha)
                    regime_train_preds.append(predict_weighted_ridge(model, X_regime.loc[train_mask]))
                    regime_valid_preds.append(predict_weighted_ridge(model, X_regime.loc[valid_mask]))
                regime_train_matrix = np.column_stack(regime_train_preds)
                regime_valid_matrix = np.column_stack(regime_valid_preds)
                blended_train_delta = np.sum(train_probs * regime_train_matrix, axis=1)
                blended_valid_delta = np.sum(valid_probs * regime_valid_matrix, axis=1)
                for schedule_name, schedule in blend_schedules.items():
                    weights = blend_weights(data.loc[valid_mask], fold_col, schedule)
                    pred_delta = (1.0 - weights) * global_valid_delta + weights * blended_valid_delta
                    pred_tvt = base_tvt + pred_delta
                    label = f"{global_name}__regime_ridge_alpha{int(alpha)}__{schedule_name}"
                    metrics = metric_row(actual, pred_tvt, base_tvt)
                    fold_rows.append({
                        "run_id": RUN_ID,
                        "fold_column": fold_col,
                        "fold": int(fold),
                        "label": label,
                        "global_model": global_name,
                        "regime_model": f"regime_ridge_alpha{int(alpha)}",
                        "schedule": schedule_name,
                        "n_train_rows": int(train_mask.sum()),
                        "n_valid_rows": int(valid_mask.sum()),
                        "n_base_features": int(len(BASE_FEATURE_COLS)),
                        "n_regime_features": int(len(REGIME_FEATURE_COLS)),
                        **metrics,
                    })
                    tmp = data.loc[valid_mask, ["wellname", PRIMARY_FOLD, STRESS_FOLD, "eval_progress", "selected_analog_distance", "combined_stress_score", f"mix_entropy__{fold_col}", f"mix_prob_margin__{fold_col}", f"mix_confidence__{fold_col}"]].copy()
                    tmp["fold_column"] = fold_col
                    tmp["fold"] = int(fold)
                    tmp["label"] = label
                    tmp["global_model"] = global_name
                    tmp["regime_model"] = f"regime_ridge_alpha{int(alpha)}"
                    tmp["schedule"] = schedule_name
                    tmp["actual_tvt"] = actual
                    tmp["pred_tvt"] = pred_tvt
                    tmp["baseline_tvt"] = base_tvt
                    tmp["global_pred_tvt"] = base_tvt + global_valid_delta
                    tmp["regime_pred_tvt"] = base_tvt + blended_valid_delta
                    tmp["blend_weight"] = weights
                    tmp["residual"] = actual - pred_tvt
                    pred_parts.append(tmp)
                    diag_rows.append({
                        "fold_column": fold_col,
                        "fold": int(fold),
                        "label": label,
                        "global_model": global_name,
                        "regime_model": f"regime_ridge_alpha{int(alpha)}",
                        "schedule": schedule_name,
                        "mean_blend_weight": float(np.mean(weights)),
                        "max_blend_weight": float(np.max(weights)),
                        "train_global_rmse": rmse(y_delta.loc[train_mask], global_train_delta),
                        "train_regime_rmse": rmse(y_delta.loc[train_mask], blended_train_delta),
                    })
    return pd.DataFrame(fold_rows), pd.concat(pred_parts, ignore_index=True), pd.DataFrame(diag_rows)


start = time.time()
strict_fold_metrics, strict_predictions, strict_diag = evaluate_regime_mean(PRIMARY_FOLD)
stress_fold_metrics, stress_predictions, stress_diag = evaluate_regime_mean(STRESS_FOLD)
print(f"elapsed_s={time.time() - start:.1f}")
strict_fold_metrics.head()


elapsed_s=293.0


,run_id,fold_column,fold,label,global_model,regime_model,schedule,n_train_rows,n_valid_rows,n_base_features,n_regime_features,rmse,mae,r2,last_known_hold_rmse
0,2026-07-01__pseudo_top_regime_mean_tuning,strict_hybrid_fold,1,xgb_square_global__regime_ridge_alpha40__s050,xgb_square_global,regime_ridge_alpha40,s050,61440,31320,41,48,13.654335,9.501366,0.999464,16.418628
1,2026-07-01__pseudo_top_regime_mean_tuning,strict_hybrid_fold,1,xgb_square_global__regime_ridge_alpha40__s065,xgb_square_global,regime_ridge_alpha40,s065,61440,31320,41,48,13.742983,9.631558,0.999457,16.418628
2,2026-07-01__pseudo_top_regime_mean_tuning,strict_hybrid_fold,1,xgb_square_global__regime_ridge_alpha40__confi...,xgb_square_global,regime_ridge_alpha40,confidence_s100,61440,31320,41,48,13.579884,9.470515,0.999469,16.418628
3,2026-07-01__pseudo_top_regime_mean_tuning,strict_hybrid_fold,1,xgb_square_global__regime_ridge_alpha40__confi...,xgb_square_global,regime_ridge_alpha40,confidence_pow050_s100,61440,31320,41,48,13.652543,9.582162,0.999464,16.418628
4,2026-07-01__pseudo_top_regime_mean_tuning,strict_hybrid_fold,1,xgb_square_global__regime_ridge_alpha40__margi...,xgb_square_global,regime_ridge_alpha40,margin_heavy_s100,61440,31320,41,48,13.596666,9.459292,0.999468,16.418628


## 7. Results And Diagnostics

In [6]:
"Summarize strict-CV, stress-CV, and regime-mean behavior."

strict_summary = (
    strict_fold_metrics.groupby(["label", "global_model", "regime_model", "schedule"])
    .agg(
        rmse_mean=("rmse", "mean"),
        rmse_std=("rmse", "std"),
        mae_mean=("mae", "mean"),
        r2_mean=("r2", "mean"),
        last_known_hold_rmse_mean=("last_known_hold_rmse", "mean"),
        n_valid_rows=("n_valid_rows", "sum"),
        n_base_features=("n_base_features", "max"),
        n_regime_features=("n_regime_features", "max"),
    )
    .reset_index()
)
stress_summary = (
    stress_fold_metrics.groupby(["label", "global_model", "regime_model", "schedule"])
    .agg(
        stress_rmse_mean=("rmse", "mean"),
        stress_rmse_std=("rmse", "std"),
        stress_mae_mean=("mae", "mean"),
        stress_last_known_hold_rmse_mean=("last_known_hold_rmse", "mean"),
        stress_n_valid_rows=("n_valid_rows", "sum"),
    )
    .reset_index()
)
summary = strict_summary.merge(stress_summary, on=["label", "global_model", "regime_model", "schedule"], how="left")
summary["rmse_delta_vs_hold"] = summary["rmse_mean"] - summary["last_known_hold_rmse_mean"]
summary["rmse_delta_vs_mixture_prior"] = summary["rmse_mean"] - 13.443736843906953
summary["rmse_delta_vs_regime_mean_baseline"] = summary["rmse_mean"] - CONFIG["baseline_regime_mean_rmse"]
summary["rmse_delta_vs_safe_typewell_reference"] = summary["rmse_mean"] - 13.450635286744287
summary["rmse_delta_vs_public_best"] = summary["rmse_mean"] - CONFIG["current_public_best_rmse"]
summary["stress_minus_strict_rmse"] = summary["stress_rmse_mean"] - summary["rmse_mean"]
summary = summary.sort_values("rmse_mean").reset_index(drop=True)
best_row = summary.iloc[0].copy()
best_label = str(best_row["label"])

predictions = pd.concat([strict_predictions, stress_predictions], ignore_index=True)
predictions["progress_zone"] = pd.cut(predictions["eval_progress"], bins=[-0.001, 0.25, 0.50, 0.75, 1.001], labels=["z0", "z1", "z2", "z3"])
predictions["analog_distance_bin"] = pd.qcut(predictions["selected_analog_distance"], q=5, labels=["closest", "near", "middle", "far", "farthest"], duplicates="drop")

zone_summary = (
    predictions.groupby(["fold_column", "label", "progress_zone"], observed=True)
    .apply(lambda group: pd.Series({
        "n_rows": len(group),
        "rmse": rmse(group["actual_tvt"], group["pred_tvt"]),
        "global_rmse": rmse(group["actual_tvt"], group["global_pred_tvt"]),
        "regime_rmse": rmse(group["actual_tvt"], group["regime_pred_tvt"]),
        "baseline_rmse": rmse(group["actual_tvt"], group["baseline_tvt"]),
        "mean_blend_weight": float(group["blend_weight"].mean()),
        "mean_mix_confidence": float(group.filter(like="mix_confidence").iloc[:, 0].mean()),
    }), include_groups=False)
    .reset_index()
)
zone_summary["rmse_delta_vs_global"] = zone_summary["rmse"] - zone_summary["global_rmse"]

analog_bin_summary = (
    predictions.groupby(["fold_column", "label", "analog_distance_bin"], observed=True)
    .apply(lambda group: pd.Series({
        "n_rows": len(group),
        "n_wells": group["wellname"].nunique(),
        "rmse": rmse(group["actual_tvt"], group["pred_tvt"]),
        "global_rmse": rmse(group["actual_tvt"], group["global_pred_tvt"]),
        "regime_rmse": rmse(group["actual_tvt"], group["regime_pred_tvt"]),
        "baseline_rmse": rmse(group["actual_tvt"], group["baseline_tvt"]),
        "mean_blend_weight": float(group["blend_weight"].mean()),
    }), include_groups=False)
    .reset_index()
)
analog_bin_summary["rmse_delta_vs_global"] = analog_bin_summary["rmse"] - analog_bin_summary["global_rmse"]

well_diagnostics = (
    predictions.loc[(predictions["fold_column"].eq(PRIMARY_FOLD)) & (predictions["label"].eq(best_label))]
    .groupby("wellname")
    .agg(
        n_rows=("actual_tvt", "size"),
        strict_fold=(PRIMARY_FOLD, "first"),
        analog_fold=(STRESS_FOLD, "first"),
        mean_selected_analog_distance=("selected_analog_distance", "mean"),
        mean_blend_weight=("blend_weight", "mean"),
        mean_residual=("residual", "mean"),
        mean_abs_residual=("residual", lambda series: float(np.abs(series).mean())),
        rmse=("residual", lambda series: float(np.sqrt(np.mean(np.square(series))))),
    )
    .reset_index()
    .sort_values("rmse", ascending=False)
)

regime_diagnostics = pd.concat([strict_diag, stress_diag], ignore_index=True)
regime_summary = (
    regime_diagnostics.groupby(["fold_column", "label", "global_model", "regime_model", "schedule"])
    .agg(
        mean_blend_weight=("mean_blend_weight", "mean"),
        max_blend_weight=("max_blend_weight", "max"),
        train_global_rmse=("train_global_rmse", "mean"),
        train_regime_rmse=("train_regime_rmse", "mean"),
    )
    .reset_index()
)

print(summary.head(18).to_string(index=False))

                                                          label           global_model          regime_model          schedule  rmse_mean  rmse_std  mae_mean  r2_mean  last_known_hold_rmse_mean  n_valid_rows  n_base_features  n_regime_features  stress_rmse_mean  stress_rmse_std  stress_mae_mean  stress_last_known_hold_rmse_mean  stress_n_valid_rows  rmse_delta_vs_hold  rmse_delta_vs_mixture_prior  rmse_delta_vs_regime_mean_baseline  rmse_delta_vs_safe_typewell_reference  rmse_delta_vs_public_best  stress_minus_strict_rmse
       xgb_square_global__regime_ridge_alpha40__confidence_s100      xgb_square_global  regime_ridge_alpha40   confidence_s100  13.090965  0.479832  9.359701 0.999548                  15.571277         92760               41                 48         13.406457         0.701691         9.746599                         15.581239                92760           -2.480312                    -0.352771                           -0.001818                              -0.3596

## 8. Artifact Export

In [7]:
"Export artifacts, report, and model-comparison row."

best_strict_rmse = float(best_row["rmse_mean"])
best_stress_rmse = float(best_row["stress_rmse_mean"])
best_global = summary.loc[summary["schedule"].eq("s050")].sort_values("rmse_mean").iloc[0]
best_confidence = summary.loc[summary["schedule"].str.contains("confidence", regex=False)].sort_values("rmse_mean").iloc[0]

if best_strict_rmse < CONFIG["baseline_regime_mean_rmse"] - 0.05 and best_stress_rmse <= CONFIG["baseline_regime_mean_stress_rmse"]:
    recommendation = "package_pseudo_top_regime_mean_tuning"
elif best_strict_rmse < CONFIG["current_public_best_rmse"]:
    recommendation = "iterate_before_package"
else:
    recommendation = "diagnostic_only"

findings = pd.DataFrame([
    {"finding": "best_overall", "detail": best_label, "value": best_strict_rmse, "interpretation": f"Best strict RMSE with analog stress RMSE {best_stress_rmse:.4f}."},
    {"finding": "best_fixed_blend", "detail": str(best_global["label"]), "value": float(best_global["rmse_mean"]), "interpretation": "Best fixed fallback/blend candidate."},
    {"finding": "best_confidence_blend", "detail": str(best_confidence["label"]), "value": float(best_confidence["rmse_mean"]), "interpretation": "Best confidence-gated regime mean candidate."},
    {"finding": "decision", "detail": recommendation, "value": best_strict_rmse, "interpretation": "Promotion rule checks mixture-prior reference, stress behavior, and current public context."},
])

paths = {
    "summary": TABLE_DIR / "model__pseudo_top_regime_mean_tuning_summary__2026-07-01.csv",
    "fold_metrics": TABLE_DIR / "model__pseudo_top_regime_mean_tuning_fold_metrics__2026-07-01.csv",
    "regime_diagnostics": TABLE_DIR / "model__pseudo_top_regime_mean_tuning_regime_diagnostics__2026-07-01.csv",
    "zone_summary": TABLE_DIR / "model__pseudo_top_regime_mean_tuning_zone_summary__2026-07-01.csv",
    "analog_bin_summary": TABLE_DIR / "model__pseudo_top_regime_mean_tuning_analog_bin_summary__2026-07-01.csv",
    "well_diagnostics": TABLE_DIR / "model__pseudo_top_regime_mean_tuning_well_diagnostics__2026-07-01.csv",
    "findings": TABLE_DIR / "model__pseudo_top_regime_mean_tuning_findings__2026-07-01.csv",
}
exports = {
    "summary": summary,
    "fold_metrics": pd.concat([strict_fold_metrics, stress_fold_metrics], ignore_index=True),
    "regime_diagnostics": regime_summary,
    "zone_summary": zone_summary,
    "analog_bin_summary": analog_bin_summary,
    "well_diagnostics": well_diagnostics,
    "findings": findings,
}
for key, frame in exports.items():
    frame.to_csv(paths[key], index=False)
    shutil.copy2(paths[key], RUN_DIR / paths[key].name)

comparison_path = TABLE_DIR / "model_comparison.csv"
comparison = pd.read_csv(comparison_path) if comparison_path.exists() else pd.DataFrame()
new_row = pd.DataFrame([{
    "run_id": RUN_ID,
    "model_family": "bayesian",
    "model_name": "pseudo_top_regime_mean_tuning",
    "best_candidate": best_label,
    "local_cv_rmse": best_strict_rmse,
    "local_cv_mae": float(best_row["mae_mean"]),
    "public_lb_rmse": np.nan,
    "status": recommendation,
    "notes": f"stress_rmse={best_stress_rmse:.4f}; best_confidence={float(best_confidence['rmse_mean']):.4f}",
    "updated_at": RUN_DATE,
}])
if not comparison.empty and "run_id" in comparison.columns:
    comparison = comparison.loc[comparison["run_id"] != RUN_ID].copy()
comparison = pd.concat([comparison, new_row], ignore_index=True, sort=False)
comparison.to_csv(comparison_path, index=False)
shutil.copy2(comparison_path, RUN_DIR / comparison_path.name)

report_path = REPORT_DIR / "model__pseudo_top_regime_mean_tuning__2026-07-01.md"
report = f"""# Pseudo-Top Regime Mean Tuning

## Result

Best overall candidate: `{best_label}` with strict-CV RMSE {best_strict_rmse:.4f}, MAE {float(best_row["mae_mean"]):.4f}, and analog-distance stress RMSE {best_stress_rmse:.4f}.

Best confidence-gated regime mean candidate: `{best_confidence["label"]}` with RMSE {float(best_confidence["rmse_mean"]):.4f}.

## Model Structure

Pseudo-top probabilities define soft formation regimes. Each regime gets a weighted Ridge MAP mean over legal row/typewell/state features. Predictions blend the regime-specific mean with a global safe/typewell mean, using either fixed blend weights or mixture-confidence weights.

## Leakage Notes

Direct formation columns are used only as fold-local labels for pseudo-top proxy training. Regime probabilities and regime means use held-out pseudo-top predictions and hidden-test-safe features.

## Decision

Recommendation: `{recommendation}`.

## Next Move

If this does not beat the first regime-mean baseline by a useful margin, move to boosted stacking or pseudo-top inference tuning rather than widening the same grid again.
"""
report_path.write_text(report, encoding="utf-8")
shutil.copy2(report_path, RUN_DIR / report_path.name)

print(findings.to_string(index=False))
print(report_path)

              finding                                                   detail     value                                                                              interpretation
         best_overall xgb_square_global__regime_ridge_alpha40__confidence_s100 13.090965                                           Best strict RMSE with analog stress RMSE 13.4065.
     best_fixed_blend            xgb_square_global__regime_ridge_alpha40__s050 13.117647                                                        Best fixed fallback/blend candidate.
best_confidence_blend xgb_square_global__regime_ridge_alpha40__confidence_s100 13.090965                                                Best confidence-gated regime mean candidate.
             decision                                   iterate_before_package 13.090965 Promotion rule checks mixture-prior reference, stress behavior, and current public context.
C:\Users\sardo\OneDrive\Desktop\Classes\Projects\Wellbore-tvt-Modeling\artifacts\reports\model_

## 9. Findings

In [8]:
"Show headline result tables."

display(summary.head(15))
display(regime_summary.loc[regime_summary["label"].eq(best_label)].head())
display(findings)

,label,global_model,regime_model,schedule,rmse_mean,rmse_std,mae_mean,r2_mean,last_known_hold_rmse_mean,n_valid_rows,n_base_features,n_regime_features,stress_rmse_mean,stress_rmse_std,stress_mae_mean,stress_last_known_hold_rmse_mean,stress_n_valid_rows,rmse_delta_vs_hold,rmse_delta_vs_mixture_prior,rmse_delta_vs_regime_mean_baseline,rmse_delta_vs_safe_typewell_reference,rmse_delta_vs_public_best,stress_minus_strict_rmse
0,xgb_square_global__regime_ridge_alpha40__confi...,xgb_square_global,regime_ridge_alpha40,confidence_s100,13.090965,0.479832,9.359701,0.999548,15.571277,92760,41,48,13.406457,0.701691,9.746599,15.581239,92760,-2.480312,-0.352771,-0.001818,-0.359670,-1.381035,0.315491
1,xgb_square_global__regime_ridge_alpha80__confi...,xgb_square_global,regime_ridge_alpha80,confidence_s100,13.092784,0.483265,9.359568,0.999548,15.571277,92760,41,48,13.400575,0.685154,9.742501,15.581239,92760,-2.478493,-0.350953,0.000000,-0.357852,-1.379216,0.307791
2,xgb_square_global__regime_ridge_alpha40__margi...,xgb_square_global,regime_ridge_alpha40,margin_heavy_s100,13.093396,0.481221,9.340972,0.999548,15.571277,92760,41,48,13.409108,0.666704,9.741504,15.581239,92760,-2.477881,-0.350341,0.000612,-0.357240,-1.378604,0.315712
3,xgb_square_global__regime_ridge_alpha80__margi...,xgb_square_global,regime_ridge_alpha80,margin_heavy_s100,13.095639,0.483983,9.341535,0.999548,15.571277,92760,41,48,13.405885,0.654667,9.738847,15.581239,92760,-2.475638,-0.348098,0.002855,-0.354996,-1.376361,0.310246
4,xgb_square_global__regime_ridge_alpha140__conf...,xgb_square_global,regime_ridge_alpha140,confidence_s100,13.096751,0.487803,9.360559,0.999548,15.571277,92760,41,48,13.394994,0.665733,9.738027,15.581239,92760,-2.474526,-0.346986,0.003967,-0.353885,-1.375249,0.298244
5,xgb_square_global__regime_ridge_alpha140__marg...,xgb_square_global,regime_ridge_alpha140,margin_heavy_s100,13.099922,0.487595,9.343182,0.999547,15.571277,92760,41,48,13.403470,0.640689,9.736294,15.581239,92760,-2.471355,-0.343815,0.007138,-0.350713,-1.372078,0.303548
6,xgb_square_global__regime_ridge_alpha220__conf...,xgb_square_global,regime_ridge_alpha220,confidence_s100,13.102984,0.492999,9.362796,0.999547,15.571277,92760,41,48,13.390835,0.645655,9.733762,15.581239,92760,-2.468293,-0.340753,0.010200,-0.347651,-1.369016,0.287851
7,xgb_square_global__regime_ridge_alpha220__marg...,xgb_square_global,regime_ridge_alpha220,margin_heavy_s100,13.106264,0.491713,9.346022,0.999547,15.571277,92760,41,48,13.402593,0.626370,9.734163,15.581239,92760,-2.465013,-0.337473,0.013480,-0.344371,-1.365736,0.296329
8,xgb_square_global__regime_ridge_alpha40__s050,xgb_square_global,regime_ridge_alpha40,s050,13.117647,0.520790,9.368244,0.999546,15.571277,92760,41,48,13.398749,0.719486,9.724025,15.581239,92760,-2.453630,-0.326090,0.024863,-0.332988,-1.354353,0.281102
9,xgb_square_global__regime_ridge_alpha80__s050,xgb_square_global,regime_ridge_alpha80,s050,13.119564,0.524565,9.368814,0.999546,15.571277,92760,41,48,13.390060,0.699277,9.717482,15.581239,92760,-2.451712,-0.324172,0.026781,-0.331071,-1.352436,0.270496


,fold_column,label,global_model,regime_model,schedule,mean_blend_weight,max_blend_weight,train_global_rmse,train_regime_rmse
11,analog_distance_fold,xgb_square_global__regime_ridge_alpha40__confi...,xgb_square_global,regime_ridge_alpha40,confidence_s100,0.403754,0.969459,5.321482,13.184549
51,strict_hybrid_fold,xgb_square_global__regime_ridge_alpha40__confi...,xgb_square_global,regime_ridge_alpha40,confidence_s100,0.408793,0.971574,5.353068,13.196643


,finding,detail,value,interpretation
0,best_overall,xgb_square_global__regime_ridge_alpha40__confi...,13.090965,Best strict RMSE with analog stress RMSE 13.4065.
1,best_fixed_blend,xgb_square_global__regime_ridge_alpha40__s050,13.117647,Best fixed fallback/blend candidate.
2,best_confidence_blend,xgb_square_global__regime_ridge_alpha40__confi...,13.090965,Best confidence-gated regime mean candidate.
3,decision,iterate_before_package,13.090965,"Promotion rule checks mixture-prior reference,..."
